# 📗 부록: 벡터DB 를 직접 만들기

**이 노트북은 수업 시간에 다루지 않는 참고 자료입니다.** 완주 기준에도 들어가지 않습니다.

교안 01 은 첫 준비 셀에서 벡터DB 를 **열기만** 했습니다 — `data/chroma_day20/` 가 이미 만들어져 함께 배포되기 때문입니다. 74쪽을 조각내 임베딩하는 일을 커널 켤 때마다 반복하면 수업이 기다림으로 채워지니까요.

그렇다고 **어떻게 만들어졌는지 모른 채** 쓰는 것은 곤란합니다. 자르는 규칙 하나만 바꿔도 검색 결과가 달라지는데, 만드는 법을 모르면 바꿀 수도 없습니다. 이 부록이 그 과정을 처음부터 끝까지 보여 줍니다.

| 단계 | 하는 일 |
|---|---|
| 1. 원본 보기 | 무엇을 색인할 것인가 — 문서 두 건과 과제용 안내문 |
| 2. 조각내기 | 자르고 **꼬리표에 조각 번호**를 남긴다(앞뒤 문맥의 열쇠) |
| 3. 임베딩·저장 | `persist_directory` 로 **폴더에 남긴다** |
| 4. 열어서 확인 | 교안 준비 셀과 **똑같은 코드**로 열어 본다 |
| 5. 다시 만들 때 | 무엇이 함께 흔들리는지 |

> ⚠️ **이 노트북은 기본적으로 `output/chroma_rebuild/` 에 새로 만듭니다.** 배포된 `data/chroma_day20/` 를 덮어쓰지 않기 위해서입니다 — 그 폴더의 조각 id 에 교안의 설명과 과제의 채점값이 묶여 있습니다. 배포본을 정말 다시 만들 일이 생기면 아래 `TARGET_DIR` 한 줄만 바꾸면 됩니다.

> 모델(LLM)은 **한 번도 부르지 않습니다.** 임베딩 계산만 하므로 `OPENAI_API_KEY` 없이 끝까지 돌아갑니다.

In [ ]:
# [제공 코드] 이 부록이 쓰는 것들 - 이 셀은 실행만 하세요.
import shutil
from pathlib import Path

import pandas as pd
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 새로 만들 곳. 배포본을 다시 만들 때만 이 줄을 'data/chroma_day20' 로 바꿉니다.
TARGET_DIR = 'output/chroma_rebuild'
Path(TARGET_DIR).parent.mkdir(exist_ok=True)

print('새로 만들 곳:', TARGET_DIR)

---
# 1. 무엇을 색인할지 원본부터 봅니다

벡터DB 에는 **세 덩어리**가 들어 있습니다. 앞의 둘은 교안이, 마지막 하나는 과제 LV2 가 씁니다.

| 컬렉션 | 원본 | 쓰는 곳 |
|---|---|---|
| `policy_day20` | 「생성형 AI 개인정보 처리 안내서」(42쪽) | 교안 데모 |
| `std_day20` | 「개인정보 처리방침 작성지침」(32쪽) | 교안 따라하기 |
| `minwon_guide_lv2` | 구청 민원 안내문 6편 | 과제 LV2 |

앞의 둘은 **한 CSV 안에 함께** 들어 있습니다 — `문서` 열로 갈라 씁니다.

In [ ]:
# 안내서 두 건이 한 파일에 들어 있습니다 -- 한 행이 '한 쪽' 입니다.
guide_df = pd.read_csv('data/guide_docs.csv')

print('문서별 쪽 수:', guide_df.groupby('문서').size().to_dict())
print(f"한 쪽 본문 길이: 평균 {guide_df['본문'].str.len().mean():.0f}자 "
      f"/ 최장 {guide_df['본문'].str.len().max()}자")
display(guide_df[['id', '문서', '쪽', '소제목']].head(3))

In [ ]:
# 과제 LV2 가 쓰는 구청 민원 안내문 -- 한 편이 문단 넷으로 되어 있습니다.
minwon_df = pd.read_csv('data/minwon_guide.csv')

print('안내문 수:', len(minwon_df), '/ 본문 길이:',
      minwon_df['text'].str.len().min(), '~', minwon_df['text'].str.len().max(), '자')
display(minwon_df[['id', 'title']])

> **한 쪽을 통째로 넣으면 안 되는 이유**가 숫자에서 보입니다. 안내서 한 쪽이 평균 1,000자를 넘습니다 — 그 안에는 여러 이야기가 섞여 있어서, 질문과 맞는 두 문장 때문에 관계없는 열 문장까지 함께 딸려 옵니다.

---
# 2. 꼬리표에 조각 번호를 남기며 조각내기

자르는 규칙은 지난 단원에서 배운 `RecursiveCharacterTextSplitter` 그대로입니다(`chunk_size=400`·`chunk_overlap=80`, 겹침은 크기의 20%).

**새로 붙는 것은 꼬리표(metadata)입니다.** 교안이 "조각의 앞뒤" 를 꺼내 올 수 있었던 것은 여기서 번호를 남겨 두었기 때문입니다.

| 꼬리표 | 무엇 | 없으면 |
|---|---|---|
| `doc_id` | 어느 쪽에서 나왔나 | 같은 문서의 다른 조각을 찾을 수 없다 |
| `chunk_no` | 그 쪽의 몇 번째 조각인가 | **앞뒤를 지목할 수 없다** |
| `chunk_total` | 그 쪽이 몇 조각인가 | 문서 끝을 넘어가는 번호를 요구하게 된다 |
| `쪽` · `소제목` | 출처 | 답에 근거를 붙일 수 없다 |

In [ ]:
# 한 문서를 조각 목록으로 바꾸는 함수 -- 세 컬렉션이 이 함수를 함께 씁니다.
#  파일마다 열 이름이 달라서(본문/text, 소제목/title) 열 이름을 인자로 받습니다.
def to_chunks(df, text_column, title_column, size, overlap, with_page):
    """DataFrame 의 각 행을 잘라 Document 목록으로 만든다(꼬리표에 조각 번호를 남긴다)."""
    splitter = RecursiveCharacterTextSplitter(chunk_size=size, chunk_overlap=overlap)
    chunks = []
    for _, row in df.iterrows():              # 한 행 = 안내서 한 쪽(또는 안내문 한 편)
        parts = splitter.split_text(row[text_column])
        for i, part in enumerate(parts):
            tag = {'doc_id': row['id'],
                   'chunk_no': i,             # 이 쪽의 몇 번째 조각인가
                   'chunk_total': len(parts), # 이 쪽이 모두 몇 조각인가
                   'title': row[title_column]}
            if with_page:                      # 안내서는 쪽·소제목이 곧 출처다
                tag['쪽'] = int(row['쪽'])     # 꼬리표에는 파이썬 기본 자료형만 담는다
                tag['소제목'] = row['소제목']
            chunks.append(Document(page_content=part, metadata=tag))
    return chunks


policy_chunks = to_chunks(guide_df[guide_df['문서'] == '생성형AI 안내서'],
                          '본문', '소제목', 400, 80, with_page=True)
std_chunks = to_chunks(guide_df[guide_df['문서'] == '처리방침 표준안'],
                       '본문', '소제목', 400, 80, with_page=True)
# 민원 안내문은 한 편이 짧아 더 잘게 자릅니다(200/40) -- 그래야 앞뒤 문맥 연습이 됩니다.
minwon_chunks = to_chunks(minwon_df, 'text', 'title', 200, 40, with_page=False)

for name, chunks in [('policy', policy_chunks), ('std', std_chunks), ('minwon', minwon_chunks)]:
    lengths = [len(chunk.page_content) for chunk in chunks]
    average = sum(lengths) // len(lengths)
    print(f'{name:7} 조각 {len(chunks):3}개 · 평균 {average}자 · 최장 {max(lengths)}자')

In [ ]:
# 조각 하나를 열어 봅니다 -- 꼬리표에 번호가 들어 있는 것이 핵심입니다.
sample = policy_chunks[1]

print('꼬리표:', sample.metadata)
print('본문  :', repr(sample.page_content[:80]))

> 조각 id 는 저장할 때 **`문서id-조각번호`**(예: `ai6-1`)로 붙입니다. 같은 id 는 덮어쓰기가 되므로 이 노트북을 여러 번 실행해도 조각이 쌓이지 않습니다. 검색 평가(교안 02)의 정답 라벨도 이 id 를 가리킵니다 — **자르는 규칙을 바꾸면 그 라벨이 통째로 어긋납니다.**

---
# 3. 임베딩해서 폴더에 남기기

여기가 시간이 걸리는 단계입니다 — 조각 300여 개를 전부 벡터로 바꿉니다(그래서 미리 만들어 두는 것입니다). 새로 배우는 인자는 하나뿐입니다.

| 인자 | 하는 일 |
|---|---|
| `persist_directory` | 색인을 **폴더에 남긴다** — 커널을 꺼도 사라지지 않는다 |
| `collection_name` | 한 폴더 안에서 색인을 구분하는 이름 |
| `ids` | 조각마다 고유 이름 — 같은 id 는 덮어쓰기 |

> 임베딩 모델은 교안이 **검색할 때 쓰는 것과 같아야 합니다.** 다르면 에러 없이 순위만 틀어집니다 — 교안 5절에서 본 **조용한 실패**의 한 종류입니다.

In [ ]:
# 이 셀이 오래 걸립니다 -- 조각 300여 개를 벡터로 바꿉니다(모델 내려받기 포함 몇 분).
# 폴더가 남아 있으면 통째로 지우고 다시 만듭니다 -- 조각이 겹쳐 쌓이지 않게.
if Path(TARGET_DIR).exists():
    shutil.rmtree(TARGET_DIR)

embeddings = HuggingFaceEmbeddings(model_name='jhgan/ko-sroberta-multitask')   # 교안이 검색할 때 쓰는 그 모델

for collection, chunks in [('policy_day20', policy_chunks),
                           ('std_day20', std_chunks),
                           ('minwon_guide_lv2', minwon_chunks)]:
    Chroma.from_documents(
        chunks, embeddings, collection_name=collection,
        persist_directory=TARGET_DIR,
        ids=[f"{d.metadata['doc_id']}-{d.metadata['chunk_no']}" for d in chunks])
    print(f'{collection:18} 조각 {len(chunks)}개 저장 완료')

---
# 4. 교안 준비 셀과 똑같은 코드로 열어서 확인

이제 만든 것을 열어 봅니다. 아래 코드는 **교안 01 의 준비 셀과 같습니다** — 다른 것은 `persist_directory` 가 방금 만든 폴더라는 것뿐입니다.

In [ ]:
# 교안 준비 셀과 같은 코드 -- 만드는 코드가 아니라 '여는' 코드입니다.
policy_store = Chroma(persist_directory=TARGET_DIR,
                      collection_name='policy_day20',
                      embedding_function=embeddings)
policy_retriever = policy_store.as_retriever(search_kwargs={'k': 3})
print('생성형AI 안내서 조각 수:', len(policy_store.get()['ids']))

std_store = Chroma(persist_directory=TARGET_DIR,
                   collection_name='std_day20',
                   embedding_function=embeddings)
std_retriever = std_store.as_retriever(search_kwargs={'k': 3})
print('처리방침 작성지침 조각 수:', len(std_store.get()['ids']))

In [ ]:
# 검색이 되는지 한 번 -- 조각 번호가 꼬리표에 들어 있는 것도 함께 확인합니다.
for hit in policy_retriever.invoke('이용자 대화를 학습에 쓰려면 무엇을 알려야 하나요?'):
    tag = hit.metadata
    print(f"{tag['doc_id']}-{tag['chunk_no']}/{tag['chunk_total']} "
          f"{tag['쪽']}쪽 · {tag['소제목'][:24]}")

> 배포된 벡터DB 를 열 때는 `TARGET_DIR` 자리에 **`data/chroma_day20`** 를 넣으면 됩니다. 교안 01 의 준비 셀이 하는 일이 정확히 그것입니다.

---
# 5. 다시 만들 때 함께 흔들리는 것들

자르는 규칙을 바꾸면 조각의 **경계와 개수와 id** 가 전부 달라집니다. 그러면 이 벡터DB 만 바뀌는 것이 아니라 **그 위에 적어 둔 것들**이 함께 어긋납니다.

| 바꾸면 | 함께 손봐야 할 것 |
|---|---|
| `chunk_size`·`chunk_overlap` | 교안 본문의 실측 수치(안내서 42쪽 → **194조각**) |
| 조각 id 규칙 | 검색 평가의 정답 라벨(`guide_eval_chunk.csv` 의 `ai33-0` 같은 값) |
| 컬렉션 이름 | 교안·과제의 준비 셀 |
| 임베딩 모델 | 검색 순위 전체(에러 없이 조용히 바뀝니다) |

그래서 배포본을 다시 만들 때는 노트북이 아니라 **빌드 스크립트 한 곳**에서 만듭니다. 규칙과 값이 여러 곳에 흩어지면 조용히 갈라지기 때문입니다.

```bash
uv run python scripts/build_day20_agent_data.py   # 표 데이터 + data/chroma_day20 을 함께 다시 만든다
```

> **한 가지만 남긴다면**: 벡터DB 는 코드가 아니라 **산출물**입니다. 산출물을 배포할 때는 *무엇으로 만들었는지*(원본·규칙·모델)를 함께 남겨야 나중에 같은 것을 다시 만들 수 있습니다. 이 부록이 그 기록입니다.